In [1]:
"""
Weather Fire Risk Modeling 
============================
Modular pipeline to predict very-low/low/moderate/high wildfire 
using weather parameters using only two SKLearn-based models and an xgboost model:
  load_data()           -> reads the CSV
  split_data()          -> 70/30 train-test split for a given target
  train_random_forest() -> fits a Random Forest (classifier or regressor)
  train_xgboost()       -> fits an XGBoost model (classifier or regressor)
  train_svm()           -> fits an SVM model (classifier or regressor)
  test_model()          -> runs predictions, returns a DataFrame of
                               actual vs. predicted values
  evaluate_models()     -> computes metrics for each model's results
  compare_models()      -> prints/returns a comparison table
  main()                -> wires everything together (No separate main function in Jupyter notebook)

LATITUDE, LONGITUDE, NEAREST_WEATHER_LAT, and NEAREST_WEATHER_LON are
excluded from model training, as is DISCOVERYDATETIME (a
raw timestamp string that isn't usable as a numeric feature as-is).
These columns are NOT dropped from the data entirely -- their test-split
values are carried through and attached to results_df alongside the
Actual/Predicted columns, so you can still see them for reference.

Requires:
    pip/conda install xgboost

NOTE ON DATA LEAKAGE:
WEATHER_RISK_CATEGORY is a deterministic binning of WEATHER_FIRE_RISK_SCORE
(High = 60.01-79.99, Moderate = 40.01-60.00, Low = 21.10-40.00,
Very Low = 15.00-20.00). Each target is excluded from the other task's
input columns so a model can't just "read off" the answer.
"""

'\nWeather Fire Risk Modeling (modular version)\n==============================================\nSame modeling task as before, refactored into functions:\n\n  load_data()           -> reads the CSV\n  split_data()          -> 70/30 train-test split for a given target\n  train_random_forest() -> fits a Random Forest (classifier or regressor)\n  train_xgboost()       -> fits an XGBoost model (classifier or regressor)\n  train_svm()           -> fits an SVM model (classifier or regressor)\n  test_model()          -> runs predictions, returns a DataFrame of\n                               actual vs. predicted values\n  evaluate_models()     -> computes metrics for each model\'s results\n  compare_models()      -> prints/returns a comparison table\n  main()                -> wires everything together (No separate main function in Jupyter notebook)\n\nNOTE ON DATA LEAKAGE:\nWEATHER_RISK_CATEGORY is a deterministic binning of WEATHER_FIRE_RISK_SCORE\n(High = 60.01-79.99, Moderate = 40.01-60.0

In [19]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)
import folium
from IPython.display import display, IFrame

#Check if a software/library is installed or not
try:
    from xgboost import XGBClassifier, XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False

In [20]:
"""
Target location: Fairbanks and max_radius to consider
"""
TARGET_LAT = 64.8378 #north
TARGET_LON = -147.7164 #West
MAX_RADIUS = 250 #miles

In [21]:
RANDOM_STATE = 42
DATA_PATH = "filtered_training_data_selectedColumn_Category_together.csv"
CLASSIFICATION_TARGET = "WEATHER_RISK_CATEGORY"
REGRESSION_TARGET = "WEATHER_FIRE_RISK_SCORE"

In [22]:
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
# Columns to exclude from model training but keep tracked in the results
# (LATITUDE/LONGITUDE/NEAREST_WEATHER_LAT/NEAREST_WEATHER_LON per request;
# DISCOVERYDATETIME added because it's a raw timestamp string that breaks
# training as-is -- see note in the response).
EXCLUDE_COLS = [
   'LATITUDE', 'LONGITUDE', 'NEAREST_WEATHER_LAT', 'NEAREST_WEATHER_LON', 'DISCOVERYDATETIME',  
   #'avg_max_temp_7d', 'avg_min_temp_7d', 'avg_rh_7d', 'total_precip_7d', 'max_wind_7d', 'consec_dry_days_7d', 'avg_solar_7d', 
   #'max_temp_7d', 'min_rh_7d', 'max_solar_7d',
   #'avg_max_temp_30d', 'total_precip_30d', 'precip_anomaly_30d', 'avg_wind_30d', 'avg_rh_30d', 'days_no_rain_30d', 'avg_solar_30d', 
   #'avg_max_temp_90d', 'max_temp_90d', 'total_precip_90d', 'days_no_rain_90d', 'avg_rh_90d', 'total_solar_90d', 
   #'days_above_heat_threshold', 'days_above_wind_threshold', 'days_low_rh', 
   #'IGNITION_WEATHER_SCORE', 'SEASONAL_SCORE', 'ASPECT_SCORE', 'ELEVATION_SCORE', 'SLOPE_WIND_SCORE', 'SPREAD_RATE_SCORE',
   'WEATHER_FIRE_RISK_SCORE', 
   'WEATHER_RISK_CATEGORY'
]
# Note The commented out columns will remain in the dataframe

In [23]:
class XGBWrapper:
    """Wraps an XGBoost model so .predict() always returns labels in the
    original space (handles the int-label encoding XGBoost needs for
    classification internally, transparently to the caller)."""

    def __init__(self, model, label_encoder=None):
        self.model = model
        self.label_encoder = label_encoder

    def predict(self, X):
        preds = self.model.predict(X)
        if self.label_encoder is not None:
            preds = self.label_encoder.inverse_transform(preds)
        return preds

In [4]:
# ------------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------------
def load_data(path):
    """Reads the CSV and returns a single DataFrame."""
    df = pd.read_csv(path)
    return df

In [29]:
# ------------------------------------------------------------------
# 2. Split data
# ------------------------------------------------------------------
def split_data(df, target_col, task_type, exclude_cols=None, test_size=0.30, random_state=RANDOM_STATE):
    """Splits a dataframe into 70% train / 30% test for the given target
    column. Stratifies on the target for classification tasks.
 
    exclude_cols: columns that should NOT be used as model inputs, but
    whose test-split values are still returned (as `tracking_test`) so
    they can be reattached to the results later.
    """
    exclude_cols = [c for c in (exclude_cols or []) if c in df.columns]
 
    X_full = df.drop(columns=[target_col])  # still includes exclude_cols
    y = df[target_col]
 
    if task_type == "classification":
        try:
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state, stratify=y
            )
        except ValueError:
            print("Warning: could not stratify (a class has too few "
                  "samples). Using a plain random split.")
            X_train_full, X_test_full, y_train, y_test = train_test_split(
                X_full, y, test_size=test_size, random_state=random_state
            )
    else:
        X_train_full, X_test_full, y_train, y_test = train_test_split(
            X_full, y, test_size=test_size, random_state=random_state
        )
 
    # Keep the excluded columns' test-split values for tracking purposes
    tracking_test = X_test_full[exclude_cols].reset_index(drop=True) if exclude_cols else None
 
    # Actual model inputs -- excluded columns dropped
    X_train = X_train_full.drop(columns=exclude_cols)
    X_test = X_test_full.drop(columns=exclude_cols)
 
    return X_train, X_test, y_train, y_test, tracking_test

In [30]:
# ------------------------------------------------------------------
# 3. Model training functions (each takes the training split as input)
# ------------------------------------------------------------------
def train_random_forest(X_train, y_train, task_type):
    """Trains a Random Forest classifier or regressor."""
    if task_type == "classification":
        model = RandomForestClassifier(
            n_estimators=300, min_samples_leaf=2, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1
        )
    else:
        model = RandomForestRegressor(
            n_estimators=300, min_samples_leaf=2,
            random_state=RANDOM_STATE, n_jobs=-1
        )
    model.fit(X_train, y_train)
    return model


def train_xgboost(X_train, y_train, task_type):
    """Trains an XGBoost classifier or regressor. Returns None if xgboost
    is not installed."""
    if not XGBOOST_AVAILABLE:
        print("NOTE: xgboost is not installed (pip install xgboost) -- "
              "skipping XGBoost model.")
        return None

    if task_type == "classification":
        label_encoder = LabelEncoder()
        y_train_enc = label_encoder.fit_transform(y_train)
        model = XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, eval_metric="mlogloss", n_jobs=-1
        )
        model.fit(X_train, y_train_enc)
        return XGBWrapper(model, label_encoder)
    else:
        model = XGBRegressor(
            n_estimators=300, max_depth=6, learning_rate=0.1,
            random_state=RANDOM_STATE, n_jobs=-1
        )
        model.fit(X_train, y_train)
        return XGBWrapper(model)


def train_svm(X_train, y_train, task_type):
    """Trains an SVM classifier or regressor. Features are scaled inside
    a pipeline since SVM is distance-based."""
    if task_type == "classification":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", class_weight="balanced", random_state=RANDOM_STATE)),
        ])
    else:
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("svr", SVR(kernel="rbf")),
        ])
    model.fit(X_train, y_train)
    return model

In [31]:
# ------------------------------------------------------------------
# 4. Test model: predict on the test split, store actual vs. predicted
# ------------------------------------------------------------------
def test_model(model, X_test, y_test, model_name, tracking_df=None):
    """Runs the model on the test split and returns a DataFrame with the
    actual and predicted values side by side. If tracking_df is provided
    (e.g. LATITUDE/LONGITUDE/etc. that were excluded from training), its
    columns are attached alongside Actual/Predicted for reference."""
    y_pred = model.predict(X_test)
    results_df = pd.DataFrame({
        "Actual": y_test.reset_index(drop=True),
        "Predicted": y_pred,
    })
    results_df["Model"] = model_name

    if tracking_df is not None:
        results_df = pd.concat(
            [tracking_df.reset_index(drop=True), results_df], axis=1
        )
        
    return results_df

In [41]:
# ----------------------------------------------------------------------------------
# Helper plot function (takes a dataframe and plot it on a map based on the lat,lon) 
# ----------------------------------------------------------------------------------
def create_map(df, lat_col='LATITUDE', lon_col='LONGITUDE', zoom_start=1,
               center_lat=TARGET_LAT, center_lon=TARGET_LON,
                save_path=None, show_in_notebook=True):
    """
    Create an interactive Folium map with red circle markers for each row
    in the dataframe. Hovering over a circle shows a tooltip with all
    column headers and their corresponding values for that record.

    Parameters:
        df (pd.DataFrame): DataFrame containing at least LATITUDE and LONGITUDE columns.
        lat_col (str): Name of the latitude column.
        lon_col (str): Name of the longitude column.
        zoom_start (int): Initial zoom level of the map.
        save_path (str or None): If provided, saves the map to this HTML file path.
        show_in_notebook (bool): If True, displays the map inline in a Jupyter notebook.

    Returns:
        folium.Map: The generated map object.
    """
    # Drop rows with missing coordinates
    df = df.dropna(subset=[lat_col, lon_col])

    # Center the map on the mean lat/lon
    center_lat = df[lat_col].mean()
    center_lon = df[lon_col].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    for _, row in df.iterrows():
        # Build an HTML table of all column headers and values for this row
        rows_html = "".join(
            f"<tr><td style='padding:2px 6px;font-weight:bold;'>{col}</td>"
            f"<td style='padding:2px 6px;'>{row[col]}</td></tr>"
            for col in df.columns
        )
        tooltip_html = f"<table>{rows_html}</table>"

        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=2,
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            tooltip=folium.Tooltip(tooltip_html, sticky=True)
        ).add_to(m)

    # Optionally save to an HTML file
    if save_path:
        m.save(save_path)

    # Optionally display inline in a Jupyter notebook
    if show_in_notebook:
        display(m)

    return m

In [42]:
# ------------------------------------------------------------------
# 5. Evaluate models: compute metrics from each model's results DataFrame
# ------------------------------------------------------------------
def evaluate_models(results_dict, task_type):
    """Takes a dict of {model_name: results_dataframe} (as produced by
    test_model) and returns a comparison DataFrame of evaluation metrics."""
    rows = []

    for model_name, results_df in results_dict.items():
        y_true = results_df["Actual"]
        y_pred = results_df["Predicted"]

        if task_type == "classification":
            rows.append({
                "Model": model_name,
                "Accuracy (%)": accuracy_score(y_true, y_pred) * 100,
                "Precision (macro)": precision_score(y_true, y_pred, average="macro", zero_division=0),
                "Recall (macro)": recall_score(y_true, y_pred, average="macro", zero_division=0),
                "F1-score (macro)": f1_score(y_true, y_pred, average="macro", zero_division=0),
                "Precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
                "Recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
                "F1-score (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            })
        else: #Not Needed Now
            mape = mean_absolute_percentage_error(y_true, y_pred)
            rows.append({
                "Model": model_name,
                "R2": r2_score(y_true, y_pred),
                "MAE": mean_absolute_error(y_true, y_pred),
                "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
                "Percentage Accuracy (100-MAPE)": (1 - mape) * 100,
            })

    comparison_df = pd.DataFrame(rows).set_index("Model").round(4)
    return comparison_df


In [43]:
# ------------------------------------------------------------------
# 6. Compare models: display (and optionally save) the comparison table
# ------------------------------------------------------------------
def compare_models(comparison_df, title, save_path=None):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)
    print(comparison_df)
    if save_path:
        comparison_df.to_csv(save_path)
        print(f"Saved comparison table to {save_path}")
    return comparison_df

In [44]:
# --- Load data ---
df = load_data(DATA_PATH)

# Exclude each target from the other task's inputs (see leakage note above)
df_for_classification = df.drop(columns=[REGRESSION_TARGET])
df_for_regression = df.drop(columns=[CLASSIFICATION_TARGET])

In [45]:
# --- Split data ---
Xc_train, Xc_test, yc_train, yc_test, tracking_c_test = split_data(
    df_for_classification, CLASSIFICATION_TARGET, task_type="classification",
    exclude_cols=EXCLUDE_COLS,
)
Xr_train, Xr_test, yr_train, yr_test, tracking_r_test = split_data(
    df_for_regression, REGRESSION_TARGET, task_type="regression", 
    exclude_cols=EXCLUDE_COLS,
)

In [46]:
# --- Train models: classification ---
rf_clf = train_random_forest(Xc_train, yc_train, task_type="classification")
xgb_clf = train_xgboost(Xc_train, yc_train, task_type="classification")
svm_clf = train_svm(Xc_train, yc_train, task_type="classification")

In [ ]:
# --- Test models: classification ---
clf_results = {}
clf_results["Random Forest"] = test_model(rf_clf, Xc_test, yc_test, "Random Forest", tracking_df=tracking_c_test)
if xgb_clf is not None:
    clf_results["XGBoost"] = test_model(xgb_clf, Xc_test, yc_test, "XGBoost", tracking_df=tracking_c_test)
clf_results["SVM"] = test_model(svm_clf, Xc_test, yc_test, "SVM", tracking_df=tracking_c_test)

#Save the results into a csv file
clf_results['Random Forest'].to_csv('clf_results_randomforest.csv', index=False)
clf_results['XGBoost'].to_csv('clf_results_xgboost.csv', index=False)
clf_results['SVM'].to_csv('clf_results_svm.csv', index=False)

#Plot the data for easy visualization
m = create_map(clf_results['Random Forest'], save_path='randomforest_predicted_vs_actual.html', zoom_start=3)
m = create_map(clf_results['XGBoost'], save_path='xgboost_predicted_vs_actual.html', zoom_start=3)
m = create_map(clf_results['SVM'], save_path='svm_predicted_vs_actual.html', zoom_start=3)

In [ ]:
# --- Evaluate + compare: classification ---
clf_comparison = evaluate_models(clf_results, task_type="classification")
compare_models(
    clf_comparison,
    "CLASSIFICATION MODEL COMPARISON (target: WEATHER_RISK_CATEGORY)",
    save_path="ml_classification_model_comparison_TraditionalML.csv",
)

In [ ]:
# --- Train models: regression ---
rf_reg = train_random_forest(Xr_train, yr_train, task_type="regression")
xgb_reg = train_xgboost(Xr_train, yr_train, task_type="regression")
svm_reg = train_svm(Xr_train, yr_train, task_type="regression")

In [ ]:
# --- Test models: regression ---
reg_results = {}
reg_results["Random Forest"] = test_model(rf_reg, Xr_test, yr_test, "Random Forest")
if xgb_reg is not None:
    reg_results["XGBoost"] = test_model(xgb_reg, Xr_test, yr_test, "XGBoost")
reg_results["SVM"] = test_model(svm_reg, Xr_test, yr_test, "SVM")

#Save the results into a csv file
reg_results['Random Forest'].to_csv('reg_results_randomforest.csv', index=False)
reg_results['XGBoost'].to_csv('reg_results_xgboost.csv', index=False)
reg_results['SVM'].to_csv('reg_results_svm.csv', index=False)

#Plot the data for easy visualization
m = create_map(reg_results['Random Forest'], save_path='randomforest_predicted_vs_actual.html', zoom_start=3)
m = create_map(reg_results['XGBoost'], save_path='xgboost_predicted_vs_actual.html', zoom_start=3)
m = create_map(reg_results['SVM'], save_path='svm_predicted_vs_actual.html', zoom_start=3)

In [ ]:
# --- Evaluate + compare: regression ---
reg_comparison = evaluate_models(reg_results, task_type="regression")
compare_models(
    reg_comparison,
    "REGRESSION MODEL COMPARISON (target: WEATHER_FIRE_RISK_SCORE)",
    save_path="regression_model_comparison_TraditionalML.csv",
)